# Módulo métricas de congestión
Fórmulas del CEJA (*I Índice de Congestión Judicial en las Américas*, 2025, cap. I):

| indicador | fórmula |
|---|---|
| tasa de resolución (clearance rate) | resueltas / ingresadas **del año** |
| tasa de congestión | (ingresadas + pendientes al inicio) / resueltas = atendidas / resueltas |
| tasa de pendencia | pendientes_fin / atendidas |
| duración estimada (días) | pendientes_fin / resueltas × 365 (estimador propio: el CEJA no define una duración) |

`ingresadas` es la **suma de todas las formas de ingreso del año**, no solo `nuevas_ingresadas`, y **no incluye el
stock inicial**. Se verifica con la identidad del anuario: ingresadas = atendidas − pendientes_inicio.

**Stock inicial en civil.** Los cuadros civiles (5.1.1.1 y 6.1.1.1) no publican `pendientes_inicio`: en su lugar
publican `readecuadas_ley_439`, en la misma posición de la tabla donde los demás cuadros ponen «pendientes de la
gestión anterior». Es el stock inicial: suma 29.688 en capitales y 13.374 en provincias, igual al `pendientes_inicio`
del 9.1.1 y del 9.1.5, y las otras formas de ingreso suman exactamente las `ingresadas` de esos cuadros. Por eso
`readecuadas_ley_439` **no es una forma de ingreso**: alimenta `pendientes_inicio_efectivo`. El criterio anterior
(contarla como ingreso) se conserva con nombre honesto en `tasa_resolucion_con_readecuadas`.

In [ ]:
import numpy as np
import pandas as pd

# Formas de ingreso del año. `readecuadas_ley_439` NO está: es el stock inicial de civil (ver arriba).
COLUMNAS_INGRESOS = [
    "nuevas_ingresadas",
    "recibidas_excusa_recusacion",
    "preliminares_formalizados",
    "cautelares_formalizados",
    "ingresadas_conversion_acciones",
    "ingresadas_reenvio",
    "otras_formas_ingreso",
]

# En los cuadros civiles ocupa el lugar de pendientes_inicio.
COLUMNA_STOCK_CIVIL = "readecuadas_ley_439"


def a_float(serie):
    # Los tipos nullable de pandas (Int64) rompen la asignación con máscara; se pasa a float64.
    return pd.Series(serie.to_numpy(dtype="float64", na_value=np.nan), index=serie.index, dtype="float64")


def calcular_ingresos_totales(df):
    presentes = []
    for c in COLUMNAS_INGRESOS:
        if c in df.columns:
            presentes.append(c)
    return df[presentes].fillna(0).sum(axis=1)


def calcular_stock_inicial(df):
    # pendientes_inicio publicado; si la fila no lo trae (cuadros civiles), readecuadas_ley_439.
    # Devuelve el stock y de dónde salió ("publicado", "readecuadas_ley_439" o None).
    vacio = pd.Series(np.nan, index=df.index, dtype="float64")
    publicado = a_float(df["pendientes_inicio"]) if "pendientes_inicio" in df.columns else vacio
    arrastre = a_float(df[COLUMNA_STOCK_CIVIL]) if COLUMNA_STOCK_CIVIL in df.columns else vacio
    ambos = publicado.notna() & arrastre.notna()
    assert not ambos.any(), "pendientes_inicio y readecuadas_ley_439 a la vez en " + str(int(ambos.sum())) + " filas"
    stock = publicado.where(publicado.notna(), arrastre)
    fuente = pd.Series([None] * len(df), index=df.index, dtype="object")
    fuente[publicado.notna()] = "publicado"
    fuente[arrastre.notna()] = COLUMNA_STOCK_CIVIL
    return stock, fuente


def verificar_identidad_ingresos(df, tolerancia=0.0, mascara=None):
    # Devuelve solo las filas donde la suma de ingresos no cierra con atendidas - stock inicial.
    suma = calcular_ingresos_totales(df)
    stock, _ = calcular_stock_inicial(df)
    comparables = df["atendidas"].notna() & stock.notna()
    if mascara is not None:
        comparables = comparables & mascara
    identidad = df["atendidas"].astype(float) - stock
    diferencia = (identidad - suma).abs()
    descuadre = comparables & (diferencia > tolerancia)
    salida = df.loc[descuadre].copy()
    salida["ingresos_por_suma"] = suma[descuadre]
    salida["ingresos_por_identidad"] = identidad[descuadre]
    salida["diferencia_ingresos"] = diferencia[descuadre]
    return salida


def cociente(numerador, denominador, indice):
    # Divide solo donde el denominador es > 0; en el resto queda nulo.
    resultado = pd.Series(np.nan, index=indice, dtype=float)
    validos = denominador > 0
    resultado[validos] = numerador[validos] / denominador[validos]
    return resultado


def calcular_indicadores_congestion(df):
    res = df.copy()
    ingresos = a_float(calcular_ingresos_totales(res))
    res["ingresos_totales"] = ingresos
    stock, fuente_stock = calcular_stock_inicial(res)
    res["pendientes_inicio_efectivo"] = stock
    res["fuente_stock_inicial"] = fuente_stock
    atendidas = a_float(res["atendidas"])
    resueltas = a_float(res["resueltas"])
    pend_fin = a_float(res["pendientes_fin"])
    nuevas = a_float(res["nuevas_ingresadas"])

    # Sin ingresos no hay tasa de resolución: queda nula, no 1.0.
    cr = cociente(resueltas, ingresos, res.index)
    res["tasa_resolucion"] = cr
    # Criterio anterior (readecuadas contadas como ingreso). Se conserva solo para comparar:
    # el stock inicial no es un ingreso del año, así que esta variante subestima la tasa de civil.
    if COLUMNA_STOCK_CIVIL in res.columns:
        readecuadas = a_float(res[COLUMNA_STOCK_CIVIL]).fillna(0)
    else:
        readecuadas = pd.Series(0.0, index=res.index)
    res["tasa_resolucion_con_readecuadas"] = cociente(resueltas, ingresos + readecuadas, res.index)
    # Variante solo con nuevas_ingresadas, para comparar. No es el clearance rate.
    res["tasa_resolucion_solo_nuevas"] = cociente(resueltas, nuevas, res.index)
    res["tasa_congestion"] = cociente(atendidas, resueltas, res.index)

    tp = cociente(pend_fin, atendidas, res.index)
    tp[atendidas == 0] = 0.0
    res["tasa_pendencia"] = tp

    de = pd.Series(np.nan, index=res.index, dtype=float)
    validos = resueltas > 0
    de[validos] = (pend_fin[validos] / resueltas[validos]) * 365.0
    res["duracion_estimada_dias"] = de

    res["acumulacion_neta_anual"] = ingresos - resueltas

    res["flag_acumula_mora"] = (cr < 1.0).fillna(False)
    res["flag_sin_resolucion_anual"] = ((atendidas > 0) & (resueltas == 0)).fillna(False)
    res["flag_sin_movimiento"] = (atendidas == 0).fillna(False)

    if "personal_items_total" in res.columns:
        items = res["personal_items_total"].astype(float)
        res["atendidas_por_item_personal"] = np.where(items > 0, atendidas / items, np.nan)
    else:
        res["atendidas_por_item_personal"] = np.nan

    if "recurso_juzgados_publicados" in res.columns:
        juzgados = res["recurso_juzgados_publicados"].astype(float)
        res["atendidas_por_juzgado_territorio"] = np.where(juzgados > 0, atendidas / juzgados, np.nan)
    else:
        res["atendidas_por_juzgado_territorio"] = np.nan

    return res
